# 🌊 DAQathon Session 1: Data Preparation and Dataset Design

This notebook explains how scalar time-series data becomes the model-ready input used in the machine-learning notebook. We choose a dataset, inspect raw examples, build or reuse a parquet cache, select labelled rows for modelling, and create train / validation / test splits.

The workshop comes with pre-loaded datasets and prepared parquet caches, so you can run the default path without rebuilding anything. If you want to bring your own data, the same setup cells let you point at ordinary CSV files or an existing parquet cache.

By the end, you should understand what the ML notebook receives as input, why the cache exists, how the split strategies work, and which settings to change for your own labelled scalar time-series dataset.


## 🧰 One-Time Cluster Setup

Before you use these notebooks on Narval or FIR for the first time, open a cluster terminal and run:

```bash
/project/def-kmoran/shared/daqathon/envs/daqathon-ml-venv/bin/jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.


## 🧭 Notebook Roadmap

Run this notebook when you want to understand or change how data becomes ML-ready.

You will walk through:

1. **Dataset setup.** Choose one of the pre-loaded workshop datasets, or configure your own CSV/parquet data.
2. **Raw-data orientation.** Preview real sensor rows and representative labelled examples.
3. **Parquet cache creation or reuse.** Use the existing workshop cache by default, or build a cache from CSV files when needed.
4. **Reviewed-row accounting.** See which labelled rows are available for modelling and how `DATA_FRACTION` changes runtime size.
5. **Train / validation / test splitting.** Compare split strategies and choose one fixed split.
6. **Train-only subsampling.** Optionally shrink only the training split while leaving validation/test untouched.

After this notebook, open `session1_machine_learning.ipynb` to train Random Forest, k-means, CNN, and Transformer examples from the prepared cache.


## ⚙️ Configuration

The next few cells separate the main configuration ideas:

- choose a dataset preset,
- optionally override a few important dataset-specific fields,
- then adjust broad run controls like how much data to use.

On Narval/FIR, if `$SLURM_TMPDIR` is available, this notebook first stages the shared raw CSV directory and the prepared parquet cache into node-local job storage. It writes all generated outputs into a runtime directory chosen in this order:

1. `$SLURM_TMPDIR`
2. `$SCRATCH`
3. repo-local `tmp/session1_outputs`


In [ ]:
from pathlib import Path
import sys

# Find the repo root before importing utilities from scripts/.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_notebook_bootstrap import build_notebook_bootstrap_namespace

# BOOTSTRAP_STATE is the small set of paths/helpers that every notebook needs.
# We assign each value explicitly so the source of later setup variables is visible.
BOOTSTRAP_STATE = build_notebook_bootstrap_namespace(NOTEBOOK_ROOT)
NOTEBOOK_ROOT = BOOTSTRAP_STATE["NOTEBOOK_ROOT"]
SHARED_DAQATHON_ROOT = BOOTSTRAP_STATE["SHARED_DAQATHON_ROOT"]
LOCAL_CACHE_DIR = BOOTSTRAP_STATE["LOCAL_CACHE_DIR"]
SHARED_CACHE_DIR = BOOTSTRAP_STATE["SHARED_CACHE_DIR"]
setup_jsonable = BOOTSTRAP_STATE["setup_jsonable"]
show_setup_json = BOOTSTRAP_STATE["show_setup_json"]
first_existing_path = BOOTSTRAP_STATE["first_existing_path"]
first_existing_csv_dir = BOOTSTRAP_STATE["first_existing_csv_dir"]

show_setup_json(
    {
        "NOTEBOOK_ROOT": NOTEBOOK_ROOT,
        "SHARED_DAQATHON_ROOT": SHARED_DAQATHON_ROOT,
        "LOCAL_CACHE_DIR": LOCAL_CACHE_DIR,
        "SHARED_CACHE_DIR": SHARED_CACHE_DIR,
    }
)


### Dataset Presets And Custom Data

The data-preparation workflow supports four pre-loaded workshop datasets:

- `ctd_conductivity`: CTD conductivity QC labels.
- `fluorometer_turbidity`: turbidity QC labels with fluorometer/CTD/oxygen context.
- `oxygen`: oxygen concentration QC labels.
- `conductivity_plugs`: custom `ml_label` classes for conductivity-plug examples.

Each preset bundles the choices that usually need to stay together: raw-data location, cache name, target-label column, measurement-feature columns, plot columns, label meanings, and k-means feature mode. For the workshop datasets, you usually only change `DATASET_PROFILE_ID`.

If you want to change what a pre-loaded workshop preset means, edit `workshop_config/session1_dataset_profiles.py`. Shared label meanings, colours, cache fallbacks, and default measurement-column lists live in `workshop_config/session1_defaults.py`. The `scripts/session1_profiles.py` module is the resolver code that reads those config values and turns them into notebook variables.

You can also bring your own labelled scalar time-series data. The simplest custom path needs:

- one or more CSV files, or an existing parquet cache,
- a timestamp column,
- a target/label column to predict,
- one or more numeric measurement columns to use as model features,
- labels that identify good/normal rows and issue/anomaly rows.

The next code cell only selects which pre-loaded workshop preset to use. The setup-controls cell after that is where you choose between the prepared-workshop path and the custom-data path. If your data starts as ordinary CSV files, that custom path can build a simple row-level parquet cache for you. For custom CSV data, use `kmeans_feature_mode="row_level"` unless you have also prepared window-summary parquet.


In [ ]:
from scripts.session1_profiles import get_dataset_profile, profile_option_summary

# Prepared-workshop dataset selector.
# Options: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
# For participant-owned data, leave this alone and set USE_CUSTOM_DATASET = True in the next cell.
DATASET_PROFILE_ID = "ctd_conductivity"
SELECTED_DATASET_PROFILE = get_dataset_profile(DATASET_PROFILE_ID)

show_setup_json(
    {
        "selected_profile": DATASET_PROFILE_ID,
        "selected_label": SELECTED_DATASET_PROFILE["label"],
        "description": SELECTED_DATASET_PROFILE["description"],
        "target_flag": SELECTED_DATASET_PROFILE["target_flag"],
        "good_labels": SELECTED_DATASET_PROFILE.get("good_labels", [1]),
        "issue_labels": SELECTED_DATASET_PROFILE.get("issue_labels", [3, 4, 9]),
        "available_profiles": profile_option_summary(),
    }
)


### Dataset Setup Controls

Choose one setup path in the next cell. Both paths return the same `DATASET_CONFIG` dictionary, and later cells read values from that dictionary directly, such as `DATASET_CONFIG["TARGET_FLAG"]`, `DATASET_CONFIG["MEASUREMENT_COLUMNS"]`, and `DATASET_CONFIG["CACHE_BUNDLE_NAME"]`.

#### `USE_CUSTOM_DATASET`

- `False`: use one of the pre-loaded workshop datasets selected by `DATASET_PROFILE_ID` in the previous cell.
- `True`: bring your own labelled scalar dataset, either from raw CSV files or from a parquet cache you already created.

#### Prepared-workshop arguments

| Argument | What it controls |
| --- | --- |
| `dataset_profile_id` | Which pre-loaded dataset preset to use. Options are `"ctd_conductivity"`, `"fluorometer_turbidity"`, `"oxygen"`, or `"conductivity_plugs"`. |
| `shared_daqathon_root` | The shared cluster/project folder where prepared raw data and caches may live. Leave this alone unless the shared project path changes. |
| `local_cache_dir` | Repo-local cache folder used when a prepared cache exists on your machine. This is useful for laptop/offline runs. |
| `shared_cache_dir` | Shared cache folder used on FIR/Narval/Nibi when the prepared cache is already built for everyone. |

#### Custom-data arguments

| Argument | What it controls |
| --- | --- |
| `raw_data_dir` | Folder containing CSV files to convert into parquet. Use this when you want every `.csv` in one folder. Set to `None` if you are only loading an existing parquet cache. |
| `raw_csv_paths` | Exact CSV files to use instead of every CSV in a folder. Example: `["/path/file_001.csv", "/path/file_002.csv"]`. |
| `parquet_cache_dir` | Folder where the parquet cache should be written or read. If the cache already exists, point this at that folder. |
| `cache_bundle_name` | Short name/stem for the cache files, such as `"my_sensor_cache"`. Keep the same name when you reload an existing cache. |
| `time_column` | Timestamp column in your data. The workshop datasets use `"Time UTC"`; change this if your CSV uses a different timestamp column name. |
| `csv_header` | Where the CSV header lives. Use `"first_row"` for ordinary CSVs, `"auto"` for ONC-style files with metadata before the header, or an integer such as `5` for a zero-based header row. |
| `target_flag` | Label column the supervised models try to predict. For custom data this might be `"label"`, `"qc_flag"`, or another class column. |
| `measurement_columns` | Numeric sensor columns used as model features by Random Forest, CNN, Transformer, and row-level k-means. Include only columns that should be inputs to the models. |
| `optional_qc_columns` | Extra label/context columns to load for inspection, but not use directly as model features. Use `[]` if you do not need any. |
| `good_labels` | Label values treated as normal/good data. These define the negative/normal class and the good-data training rows for the forecasting anomaly example. |
| `issue_labels` | Label values treated as problem/anomaly data. These define the positive/issue rows in summaries, split checks, and supervised scoring. |
| `plot_measurement_column` | Main y-axis column for time-series plots. This is usually one of the `measurement_columns`. |
| `plot_secondary_column` | Optional second y-axis column for context plots. Use `None` if one plotted measurement is enough. |
| `kmeans_feature_mode` | How k-means gets features. Use `"row_level"` for simple custom CSV caches. Use `"window_summary"` only if you have prepared window-summary parquet too. |


In [ ]:
from scripts.session1_profiles import (
    dataset_profile_summary,
    dataset_profile_summary_rows,
    resolve_custom_dataset_config,
    resolve_workshop_dataset_config,
)

# Set this to True only when you are bringing your own labelled scalar dataset.
USE_CUSTOM_DATASET = False

if USE_CUSTOM_DATASET:
    # Custom-data path: use this outside the workshop presets, including batch jobs
    # on Nibi/Narval where you want to call the same helper from a Python script.
    DATASET_CONFIG = resolve_custom_dataset_config(
        # Use raw_data_dir for every CSV in a folder, or raw_csv_paths for an exact file list.
        # If you are loading an existing cache and do not need CSVs, set raw_data_dir=None.
        raw_data_dir="/path/to/csv_folder",
        raw_csv_paths=None,  # Example: ["/path/to/file_001.csv", "/path/to/file_002.csv"]
        # Use parquet_cache_dir to write/read the cache created from your CSV files.
        # If you already have a cache, point this at that folder and keep the same cache_bundle_name.
        parquet_cache_dir="/path/to/parquet_cache_folder",
        cache_bundle_name="my_sensor_cache",
        # CSV/header controls. Use "first_row" for ordinary CSVs, "auto" for ONC-style headers,
        # or an integer such as 5 when the header is on zero-based row 5.
        time_column="Time UTC",
        csv_header="first_row",
        # Modelling controls: target_flag is the label to predict; measurement_columns are numeric features.
        target_flag="label",
        measurement_columns=["sensor_value"],
        optional_qc_columns=[],
        # Label groups: good_labels are normal rows; issue_labels are anomaly/problem rows.
        good_labels=[0],
        issue_labels=[1],
        # Plot controls: choose one primary measurement and, optionally, one context column.
        plot_measurement_column="sensor_value",
        plot_secondary_column=None,
        # Custom CSV caches usually contain row-level data only, not precomputed window summaries.
        kmeans_feature_mode="row_level",
    )
else:
    # Prepared-workshop path: this keeps workshop-specific path detection inside the helper.
    DATASET_CONFIG = resolve_workshop_dataset_config(
        dataset_profile_id=DATASET_PROFILE_ID,
        shared_daqathon_root=SHARED_DAQATHON_ROOT,
        local_cache_dir=LOCAL_CACHE_DIR,
        shared_cache_dir=SHARED_CACHE_DIR,
    )

# DATASET_CONFIG is the resolved dataset/settings dictionary used below.
# Later cells read values from it directly, for example DATASET_CONFIG["TARGET_FLAG"].
show_setup_json({"resolved_dataset_summary": dataset_profile_summary(DATASET_CONFIG)})
show_setup_json({"resolved_dataset_settings": dataset_profile_summary_rows(DATASET_CONFIG)})


## Run Controls

These are the notebook-wide knobs you are most likely to change.

The most important is `DATA_FRACTION`:

- use a small value for quick live demos,
- increase it when you want a stronger run,
- keep it near `1.0` when you want to use the largest training budget.

This cell also keeps the early raw-data preview explicit: you can choose how many files to inspect in Part 1 without affecting the fixed split choices later on.


In [ ]:
# DATA_FRACTION is the main dataset-size control for this notebook.
# Good starting values:
# - 0.1 for a quick workshop run,
# - 0.5 for a larger but still manageable run,
# - 0.9 for a near-full run,
# - 1.0 when you want to remove the routine fraction-based row/window caps.
DATA_FRACTION = 0.1
# Keep DATA_FRACTION in (0, 1]; 0 would select no data, and values above 1
# would request more than the prepared cache contains.
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Part 1 previews only a few raw files so participants can inspect real rows quickly.
# Options: "spread" picks files across the full time range; "first" picks the earliest files.
ORIENTATION_FILE_SELECTION_MODE = "spread"
# Number of raw files to preview in Part 1. Set to None to preview every available file.
ORIENTATION_FILE_LIMIT = 3
# Maximum plotted points per example panel before visual downsampling.
# Raise this for more detail; lower it if plots render slowly.
BASE_FLAG_EXAMPLE_POINTS_PER_PANEL = 30_000

# Share of reviewed modelling rows assigned to training for each split strategy.
TRAIN_FRACTION = 0.70
# Share assigned to validation. The remaining share becomes the test split.
VALIDATION_FRACTION = 0.15

# Random seed used for reproducible sampling and stochastic model training.
# Change it when you want to see how sensitive results are to random choices.
SEED = 21

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "ORIENTATION_FILE_SELECTION_MODE": ORIENTATION_FILE_SELECTION_MODE,
        "ORIENTATION_FILE_LIMIT": ORIENTATION_FILE_LIMIT,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
        "SEED": SEED,
    }
)


In [ ]:
import copy
import importlib
import json
import os
import pickle
import subprocess
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_intro_notebook_setup import build_intro_runtime_state

# INTRO_SETUP contains only runtime/path state: output folders, staged-cache
# locations, and model artifact paths. Imports stay as normal imports below.
INTRO_SETUP = build_intro_runtime_state(
    read_raw_data_dir=DATASET_CONFIG['READ_RAW_DATA_DIR'],
    read_cache_dir=DATASET_CONFIG['READ_CACHE_DIR'],
    cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
    seed=SEED,
)

import scripts.prepare_scalar_session1_data as prepare_scalar_session1_data
import scripts.onc_scalar_cache_utils as onc_scalar_cache_utils
import scripts.parquet_cache_utils as parquet_cache_utils
import scripts.session1_intro_utils as session1_intro_utils
import scripts.session1_modeling as session1_modeling

importlib.invalidate_caches()
prepare_scalar_session1_data = importlib.reload(prepare_scalar_session1_data)
onc_scalar_cache_utils = importlib.reload(onc_scalar_cache_utils)
parquet_cache_utils = importlib.reload(parquet_cache_utils)
session1_intro_utils = importlib.reload(session1_intro_utils)
session1_modeling = importlib.reload(session1_modeling)

from scripts.session1_modeling import (
    add_temporal_context_features,
    add_tabular_baseline_features,
    apply_target_strategy,
    build_cache_bundle_paths,
    build_labeled_intervals,
    build_model_frame,
    build_reviewed_target_frame,
    build_sequence_split_bundle,
    build_sequence_split_bundle_from_frames,
    build_sequence_label_interval_data,
    build_window_classification_interval_data,
    clean_source_file_label,
    compute_contiguous_split_target_distribution,
    compute_interval_classification_metrics,
    compute_split_share_gap,
    DEFAULT_FLAG_PALETTE,
    evaluate_classifier,
    fit_extra_trees,
    fit_kmeans,
    fit_random_forest,
    infer_interval_origin,
    load_full_row_level_frame,
    load_rows_for_time_range,
    load_selected_row_level_frame,
    materialize_reviewed_split_frames,
    merge_adjacent_intervals,
    plot_cluster_window_examples,
    plot_flag_examples,
    plot_time_series_with_bands,
    predict_cnn_window_model,
    predict_sequence_label_cnn,
    predict_transformer_sequence_model,
    predict_transformer_window_model,
    resolve_cache_bundle_paths,
    resolve_runtime_output_root,
    report_average,
    run_cnn_search,
    run_cnn_search_from_frames,
    run_rf_search,
    sample_frame_by_strategy,
    scan_interleaved_block_rows,
    select_time_range,
    split_frame_by_strategy,
    stage_cache_into_runtime,
    stage_directory_into_runtime,
    summarize_split_distributions,
    summarize_target_by_time_bin,
    SUPPORTED_SPLIT_STRATEGIES,
)
from scripts.session1_intro_utils import (
    choose_cache_bundle_paths,
    directory_size_bytes,
    derive_fractional_row_limit,
    filter_csv_paths_with_required_columns,
    load_raw_flag_context_sample,
    load_raw_scalar_sample,
    build_reviewed_modelling_split,
    show_reviewed_modelling_split_build,
    show_reviewed_model_row_accounting,
    show_fixed_split_review,
    show_cnn_interval_demo,
    show_episode_aware_split_comparison,
    summarize_issue_adequacy,
    build_split_strategy_tables,
    show_session1_cache_inspection,
    load_session1_cache_context,
    read_parquet_head,
    select_part_paths,
    stage_session1_inputs,
    show_reviewed_split_summary,
    show_session1_cache_read_comparison,
    show_split_strategy_comparison,
    show_split_strategy_timeline,
    show_temporal_flag_summary,
)
from scripts.parquet_cache_utils import (
    csv_files_to_row_parquet_cache,
    resolve_csv_paths,
    resolve_or_create_parquet_cache,
)
from scripts.onc_scalar_cache_utils import (
    create_onc_row_level_parquet_cache,
    create_onc_window_summary_parquet_cache,
)
from scripts.prepare_scalar_session1_data import (
    DEFAULT_MEASUREMENT_COLUMNS,
    locate_header,
    read_scalar_csv,
)

show_setup_json(
    {
        **INTRO_SETUP["INTRO_NOTEBOOK_SETUP_SUMMARY"],
        "TASK_MODE": DATASET_CONFIG['TASK_MODE'],
        "TARGET_FLAG": DATASET_CONFIG['TARGET_FLAG'],
        "DATA_FRACTION": DATA_FRACTION,
        "SHARED_DAQATHON_ROOT": str(SHARED_DAQATHON_ROOT) if SHARED_DAQATHON_ROOT else None,
    }
)


## 🚀 Cluster Input Staging

On Narval/FIR, the shared project space is the long-lived home for the data. Interactive notebook jobs on Alliance systems usually also get a job-specific temporary directory called `$SLURM_TMPDIR`.

`SLURM_TMPDIR` is fast, local to the compute node, and deleted when the job ends. That makes it useful for repeated reads during one notebook run, but not for long-term storage.

The next code cell stages raw CSVs and the prepared parquet cache into runtime storage when `$SLURM_TMPDIR` is available, then updates the active paths so later cells read from the fast copy. Locally, or when runtime staging is disabled, it simply keeps using the original paths.


In [ ]:
stage_result = stage_session1_inputs(
    read_raw_data_dir=DATASET_CONFIG['READ_RAW_DATA_DIR'],
    runtime_raw_data_dir=INTRO_SETUP["RUNTIME_RAW_DATA_DIR"],
    read_cache_dir=DATASET_CONFIG['READ_CACHE_DIR'],
    runtime_cache_dir=INTRO_SETUP["RUNTIME_CACHE_DIR"],
    cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
    dataset_label=DATASET_CONFIG['DATASET_LABEL'],
    use_runtime_raw_data_for_reads=INTRO_SETUP["USE_RUNTIME_RAW_DATA_FOR_READS"],
    use_runtime_cache_for_reads=INTRO_SETUP["USE_RUNTIME_CACHE_FOR_READS"],
    raw_csv_paths=DATASET_CONFIG["RAW_CSV_PATHS"],
    force=False,
)

# STAGE_STATE contains the active raw/cache paths after optional SLURM_TMPDIR staging.
STAGE_STATE = stage_result["notebook_values"]
RAW_DATA_DIR = STAGE_STATE["RAW_DATA_DIR"]
RAW_CSV_PATHS = STAGE_STATE["RAW_CSV_PATHS"]
CACHE_DIR = STAGE_STATE["CACHE_DIR"]
ROW_CACHE_DIR = STAGE_STATE["ROW_CACHE_DIR"]
WINDOW_CACHE_PATH = STAGE_STATE["WINDOW_CACHE_PATH"]
METADATA_PATH = STAGE_STATE["METADATA_PATH"]
active_cache_paths = STAGE_STATE["active_cache_paths"]
print(stage_result["summary"])


## Part 1 — Orientation and Dataset

Start by understanding what the selected scalar dataset contains, what the target labels mean, and what success looks like before we touch any modelling code.


#### About The Dataset

This notebook now supports a small set of **scalar dataset presets** rather than one hard-coded instrument export.

The active preset is chosen in the configuration cell near the top. That preset tells the notebook:

- which raw-data folder to read,
- which target-label column is used,
- which measurement columns to model,
- which two variables to emphasize in plots,
- and whether k-means should cluster window summaries or row-level values.

Across all of these presets, the mental model stays the same:

- each row is one timestamp in UTC,
- measurement columns hold scalar sensor values,
- target-label columns hold the reviewed label we want the models to learn.

The printed config summary tells you exactly which preset is active for the current run.


#### What The QC Flags Mean

Many ONC scalar datasets use the standard QAQC flag system. Some presets, such as conductivity plugs, use a custom label column instead; the active target column and label meanings are shown in the setup output above. For standard QAQC flags, the official reference is:

- [ONC Quality Assurance Quality Control](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)

A practical reading of the most common flags is:

- `0`: no quality control has been applied to that value yet
- `1`: passed all tests and is treated as good data
- `2`: probably good, but worth a little more caution than `1`
- `3`: probably bad or suspect
- `4`: bad and usually treated as failed data
- `6`: insufficient valid data for reliable down-sampling
- `7`: averaged value
- `8`: interpolated value
- `9`: missing value (`NaN`)

A few practical notes are especially important for this notebook:

- `1` is the clearest "normal / healthy" label.
- `3` and `4` are the main issue labels we usually care about in supervised QC experiments.
- `9` means the value is missing, not merely noisy.
- `0` means no QC decision has been applied.

ONC's reference page also explains that manual review can override automated tests, which is useful context when you see the same instrument behaviour receive different final flags over time.

The selected target flag column is shown in the config summary, and later cells will show which QC values actually appear in the active dataset and which ones we treat as reviewed labels for modelling.


### Looking At Real Examples Early

Before moving on to caching or feature engineering, it helps to inspect a lightweight sample of the raw CSV data directly.

In the next few cells we:

- load a small sample from a few raw CSV files spread across the time range,
- look at a few real rows,
- and plot representative QC intervals in context.

This gives you a concrete picture of what one timestamp looks like before the notebook starts building model inputs from it.


### Small Utilities For Exploration Samples

A few small file-reading helpers are imported by the setup cell so this notebook can stay focused on the data itself.

That keeps the early orientation cells simple while leaving the split and train-subsampling logic visible later in Part 3.


In [ ]:
# Load a lightweight exploration sample directly from raw CSV files.
raw_csv_paths = sorted(Path(RAW_DATA_DIR).glob("*.csv"))
if not raw_csv_paths:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")

# These label-display variables come from the selected dataset profile.
# For conductivity plugs they refer to custom ml_label classes; for the other
# datasets they refer to the selected QC flag column.

orientation_required_columns = ["Time UTC", DATASET_CONFIG['FLAG_EXAMPLE_TARGET']]
if DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN']:
    orientation_required_columns.append(DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN'])
orientation_candidate_paths = filter_csv_paths_with_required_columns(
    raw_csv_paths,
    orientation_required_columns,
)
orientation_source_paths = orientation_candidate_paths if orientation_candidate_paths else raw_csv_paths

orientation_selected_paths = select_part_paths(
    orientation_source_paths,
    limit=ORIENTATION_FILE_LIMIT,
    mode=ORIENTATION_FILE_SELECTION_MODE,
)
ORIENTATION_ROWS_PER_FILE = 4000
orientation_columns = list(
    dict.fromkeys(
        [
            "Time UTC",
            DATASET_CONFIG['FLAG_EXAMPLE_TARGET'],
            DATASET_CONFIG['TARGET_FLAG'],
            *DATASET_CONFIG['OPTIONAL_QC_COLUMNS'],
            DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN'],
            DATASET_CONFIG['PLOT_SECONDARY_COLUMN'],
            *DATASET_CONFIG['MEASUREMENT_COLUMNS'],
            "Depth (m)",
        ]
    )
)

exploration_df = load_raw_scalar_sample(
    orientation_selected_paths,
    sample_rows_per_file=ORIENTATION_ROWS_PER_FILE,
    columns=orientation_columns,
)

flag_context_df = load_raw_flag_context_sample(
    orientation_selected_paths,
    target_flag=DATASET_CONFIG['FLAG_EXAMPLE_TARGET'],
    classes=DATASET_CONFIG['FLAG_EXAMPLE_CLASSES'],
    context_rows_per_class=BASE_FLAG_EXAMPLE_POINTS_PER_PANEL,
    columns=orientation_columns,
)

print(
    {
        "orientation_raw_data_dir": RAW_DATA_DIR,
        "orientation_candidate_files": len(orientation_candidate_paths),
        "orientation_selected_files": len(orientation_selected_paths),
        "orientation_rows_loaded": len(exploration_df),
        "flag_context_rows_loaded": len(flag_context_df),
        "FLAG_EXAMPLE_TARGET": DATASET_CONFIG['FLAG_EXAMPLE_TARGET'],
        "FLAG_EXAMPLE_DISPLAY_NAME": DATASET_CONFIG['FLAG_EXAMPLE_DISPLAY_NAME'],
        "FLAG_EXAMPLE_AVOID_CONTEXT_LABELS": DATASET_CONFIG['FLAG_EXAMPLE_AVOID_CONTEXT_LABELS'],
        "ORIENTATION_FILE_SELECTION_MODE": ORIENTATION_FILE_SELECTION_MODE,
        "ORIENTATION_FILE_LIMIT": ORIENTATION_FILE_LIMIT,
        "ORIENTATION_ROWS_PER_FILE": ORIENTATION_ROWS_PER_FILE,
        "orientation_preview_columns": orientation_columns,
    }
)




### Preview The Selected Raw Rows

The next cell prints a small table from the raw CSV sample. This is a quick sanity check before we talk about labels or models:

- the timestamp column should parse correctly,
- the plotting columns should contain numeric sensor values,
- the target column should be present,
- and any optional QC/context columns should appear if they exist for the active dataset.


In [ ]:
# Look at a few rows so the dataset description is tied to the actual table.
preview_candidates = [
    "Time UTC",
    DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN'],
    DATASET_CONFIG['PLOT_SECONDARY_COLUMN'],
    "Depth (m)",
    DATASET_CONFIG['TARGET_FLAG'],
    *DATASET_CONFIG['OPTIONAL_QC_COLUMNS'],
]
preview_columns = [column for column in preview_candidates if column in exploration_df.columns]
display(exploration_df[preview_columns].head(8))
print(
    {
        "time_start": exploration_df["Time UTC"].min().isoformat(),
        "time_end": exploration_df["Time UTC"].max().isoformat(),
        "preview_columns": preview_columns,
    }
)


### Seeing The Target Labels In Context

Before training anything, it helps to look at the time series directly.

The plots below show local windows centred on representative target-label values pulled directly from the raw CSV files. This is useful because:

- it connects the abstract label numbers to real sensor behaviour,
- it shows that some problematic labels appear in coherent stretches rather than isolated single rows,
- it reminds you that the model is trying to learn from patterns in time, not from labels in isolation.

The next config cell controls how those examples are drawn:

- `FLAG_EXAMPLE_POINTS_PER_PANEL` controls the maximum number of points shown in each panel. It starts from `BASE_FLAG_EXAMPLE_POINTS_PER_PANEL`, which was set in the Run Controls cell.
- `FLAG_EXAMPLE_REGION_ALPHA` controls how transparent the coloured label regions are. Larger values make label bands darker.
- `FLAG_EXAMPLE_SHOW_POINTS` switches between line-only plots and line-plus-point plots. Points are useful for sparse or irregular data.
- `FLAG_EXAMPLE_CLASSES` lists which target labels we try to show examples for.


In [ ]:
FLAG_EXAMPLE_POINTS_PER_PANEL = 5_000 if DATASET_PROFILE_ID == "conductivity_plugs" else BASE_FLAG_EXAMPLE_POINTS_PER_PANEL
FLAG_EXAMPLE_REGION_ALPHA = 0.18
FLAG_EXAMPLE_SHOW_POINTS = DATASET_PROFILE_ID == "conductivity_plugs"

print(
    {
        "FLAG_EXAMPLE_POINTS_PER_PANEL": FLAG_EXAMPLE_POINTS_PER_PANEL,
        "FLAG_EXAMPLE_REGION_ALPHA": FLAG_EXAMPLE_REGION_ALPHA,
        "FLAG_EXAMPLE_SHOW_POINTS": FLAG_EXAMPLE_SHOW_POINTS,
        "FLAG_EXAMPLE_CLASSES": DATASET_CONFIG['FLAG_EXAMPLE_CLASSES'],
    }
)




In [ ]:
# Plot representative local windows for the main target-label values we see in this dataset.
timeseries_figure, timeseries_examples = plot_flag_examples(
    flag_context_df,
    target_flag=DATASET_CONFIG['FLAG_EXAMPLE_TARGET'],
    target_display_name=DATASET_CONFIG['FLAG_EXAMPLE_DISPLAY_NAME'],
    label_meanings=DATASET_CONFIG['FLAG_EXAMPLE_LABEL_MEANINGS'],
    avoid_context_labels=DATASET_CONFIG['FLAG_EXAMPLE_AVOID_CONTEXT_LABELS'],
    measurement_column=DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN'],
    secondary_column=DATASET_CONFIG['PLOT_SECONDARY_COLUMN'],
    points_per_panel=FLAG_EXAMPLE_POINTS_PER_PANEL,
    classes=DATASET_CONFIG['FLAG_EXAMPLE_CLASSES'],
    good_labels=DATASET_CONFIG['GOOD_LABELS'],
    region_alpha=FLAG_EXAMPLE_REGION_ALPHA,
    show_flag_points=FLAG_EXAMPLE_SHOW_POINTS,
)
plt.show()
display(timeseries_examples)




## Part 2 — Data Preparation and Caching

This part explains how to turn raw ONC CSV exports into typed parquet tables that are much easier to reuse in ML experiments.


### 🧹 How We Prepare The Data

This section keeps the preparation story short and reusable:

1. start from raw ONC CSV exports,
2. clean them into typed rows,
3. save a reusable parquet cache,
4. reuse that cache for the rest of the notebook.

The detailed parsing logic lives in `scripts/onc_scalar_cache_pipeline.py`. Here we focus on the inputs, outputs, and the few knobs you are most likely to change.


### Step-By-Step: Turning Raw Data Into ML-Ready Data

A good preparation pipeline is usually simpler than it first sounds.

**1. Decide what one row should represent**

In this notebook, one row represents one timestamp from the cleaned sensor table. After preparation, that row has a parsed time, numeric measurement values, a source-file name, and the target label column we will use for modelling.

**2. Keep an explicit list of the fields you need**

For scalar time series, that usually means:

- a reliable timestamp,
- one or more measurement columns,
- the target-label column,
- and any extra QC/context columns you want to inspect.

Those choices are the reason we define `TARGET_FLAG`, `MEASUREMENT_COLUMNS`, and `OPTIONAL_QC_COLUMNS` up front.

**3. Clean once, then reuse the result**

Raw CSV files are not ideal for repeated modelling work. Each time we read them, we may need to locate the header, parse timestamps, convert columns to numeric values, drop unusable timestamps, sort by time, and remember which file each row came from.

The parquet cache saves the cleaned result so later notebook sections can reload the same prepared rows quickly and consistently.

**4. Save the views that later sections need**

We keep two possible views of the prepared data:

- **row-level parquet**: one prepared row per timestamp, useful for Random Forests, sequence models, and custom CSV data,
- **window-summary parquet**: one prepared row per fixed time window, useful when k-means should cluster short stretches summarised into feature vectors.

Workshop presets usually already have both cache views prepared. For your own CSV files, the notebook can create the row-level parquet cache automatically; if you do not prepare window summaries, use `kmeans_feature_mode="row_level"` in `resolve_custom_dataset_config(...)`.


---


### 📦 Choose The Parquet Cache Path

At this point the dataset setup has already answered one important question: **are we using a prepared workshop dataset or participant-owned data?**

The cache handling is split into two explicit paths:

- **Prepared workshop cache:** the next code cell resolves the active paths for parquet files that already exist.
- **Custom CSV cache:** if `USE_CUSTOM_DATASET = True` and your config points at CSV files, the following code cell can create a simple row-level parquet cache from those CSVs.

Both paths finish with the same shared step: load the active cache metadata so the rest of the notebook uses `ROW_CACHE_DIR`, `WINDOW_CACHE_PATH`, and `METADATA_PATH` consistently.

#### Path A: Use The Prepared Workshop Parquet Cache

Run this cell for the pre-loaded workshop datasets. It checks the active cache bundle and records the resolved row-level parquet directory, window-summary parquet path, and metadata path.

If this reports a missing cache on a cluster, the fix is usually an environment/path issue rather than a modelling issue.

In [ ]:
if DATASET_CONFIG['IS_CUSTOM_DATASET']:
    print(
        {
            "prepared_workshop_cache": "skipped",
            "reason": "The active dataset is custom; use the custom parquet cell below if CSVs need to be converted.",
        }
    )
else:
    # Prepared workshop datasets should already have parquet files in READ_CACHE_DIR
    # or in the staged runtime cache. We therefore ask the helper to *reuse only*.
    prepared_cache_result = resolve_or_create_parquet_cache(
        cache_root=Path(CACHE_DIR),
        cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
        target_flag=DATASET_CONFIG['TARGET_FLAG'],
        measurement_columns=DATASET_CONFIG['MEASUREMENT_COLUMNS'],
        optional_qc_columns=DATASET_CONFIG['OPTIONAL_QC_COLUMNS'],
        issue_labels=DATASET_CONFIG['ISSUE_LABELS'],
        # No CSV conversion happens in this path.
        raw_data_dir=None,
        raw_csv_paths=None,
        build_cache_if_missing=False,
        force_rebuild_cache=False,
        generic_csv_cache=False,
        primary_device=DATASET_CONFIG['PRIMARY_DEVICE'],
    )

    # Keep these path variables synchronized for later cells.
    active_cache_paths = prepared_cache_result["cache_paths"]
    CACHE_DIR = str(active_cache_paths.root)
    ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
    WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
    METADATA_PATH = str(active_cache_paths.metadata_path)

    print(
        {
            "prepared_workshop_cache": "resolved",
            "cache_bundle_name": DATASET_CONFIG['CACHE_BUNDLE_NAME'],
            "CACHE_DIR": CACHE_DIR,
            "ROW_CACHE_DIR": ROW_CACHE_DIR,
            "WINDOW_CACHE_PATH": WINDOW_CACHE_PATH,
            "METADATA_PATH": METADATA_PATH,
            **prepared_cache_result["summary"],
        }
    )


#### Path B: Create A Parquet Cache From A Custom CSV Dataset

Run this cell when `USE_CUSTOM_DATASET = True` and your data starts as CSV files. It creates a **row-level** parquet cache: one cleaned row per timestamp.

This path uses the generic CSV-to-row-parquet helper. The ONC-specific companion-device merge and window-summary builder live in the workshop cache utilities. For k-means with this custom row-level cache, use `kmeans_feature_mode="row_level"` in `resolve_custom_dataset_config(...)`.

If your custom data already has a parquet cache, leave `AUTO_BUILD_MISSING_CACHE = False` in the custom config and this cell will skip creation.

In [ ]:
# Set True to create the custom parquet cache when it is missing.
# Set False when you only want to load an existing custom cache.
CREATE_CUSTOM_PARQUET_IF_MISSING = True

# Set True only when you intentionally want to overwrite an existing custom cache.
# This deletes/replaces the cache bundle with the same CACHE_BUNDLE_NAME.
FORCE_REBUILD_CUSTOM_PARQUET = False

# Optional build-time limits for quick tests. None means use all CSV files/rows.
CUSTOM_PREP_MAX_FILES = None
CUSTOM_PREP_SAMPLE_ROWS = DATASET_CONFIG['DEFAULT_PREP_SAMPLE_ROWS']

if not DATASET_CONFIG['IS_CUSTOM_DATASET']:
    print(
        {
            "custom_parquet_cache": "skipped",
            "reason": "The active dataset is a prepared workshop dataset.",
        }
    )
elif not DATASET_CONFIG['AUTO_BUILD_MISSING_CACHE']:
    print(
        {
            "custom_parquet_cache": "not built",
            "reason": "AUTO_BUILD_MISSING_CACHE is False, so this run expects an existing custom cache.",
            "CACHE_DIR": CACHE_DIR,
            "CACHE_BUNDLE_NAME": DATASET_CONFIG['CACHE_BUNDLE_NAME'],
        }
    )
else:
    # Custom CSV conversion uses the generic row-level helper under the hood.
    # Required inputs come from resolve_custom_dataset_config(...):
    # - RAW_DATA_DIR / RAW_CSV_PATHS locate the CSV files,
    # - TIME_COLUMN identifies timestamps,
    # - TARGET_FLAG is the label column,
    # - MEASUREMENT_COLUMNS are numeric model features.
    custom_cache_result = resolve_or_create_parquet_cache(
        cache_root=Path(CACHE_DIR),
        cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
        raw_data_dir=RAW_DATA_DIR,
        raw_csv_paths=DATASET_CONFIG["RAW_CSV_PATHS"],
        target_flag=DATASET_CONFIG['TARGET_FLAG'],
        measurement_columns=DATASET_CONFIG['MEASUREMENT_COLUMNS'],
        optional_qc_columns=DATASET_CONFIG['OPTIONAL_QC_COLUMNS'],
        issue_labels=DATASET_CONFIG['ISSUE_LABELS'],
        time_column=DATASET_CONFIG['TIME_COLUMN'],
        csv_header=DATASET_CONFIG['CSV_HEADER'],
        build_cache_if_missing=CREATE_CUSTOM_PARQUET_IF_MISSING,
        force_rebuild_cache=FORCE_REBUILD_CUSTOM_PARQUET,
        generic_csv_cache=True,
        max_files=CUSTOM_PREP_MAX_FILES,
        sample_rows=CUSTOM_PREP_SAMPLE_ROWS,
    )

    # Keep these path variables synchronized for later cells.
    active_cache_paths = custom_cache_result["cache_paths"]
    CACHE_DIR = str(active_cache_paths.root)
    ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
    WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
    METADATA_PATH = str(active_cache_paths.metadata_path)

    print(
        {
            "custom_parquet_cache": "resolved_or_built",
            "CREATE_CUSTOM_PARQUET_IF_MISSING": CREATE_CUSTOM_PARQUET_IF_MISSING,
            "FORCE_REBUILD_CUSTOM_PARQUET": FORCE_REBUILD_CUSTOM_PARQUET,
            "CUSTOM_PREP_MAX_FILES": CUSTOM_PREP_MAX_FILES,
            "CUSTOM_PREP_SAMPLE_ROWS": CUSTOM_PREP_SAMPLE_ROWS,
            "CACHE_DIR": CACHE_DIR,
            "ROW_CACHE_DIR": ROW_CACHE_DIR,
            "METADATA_PATH": METADATA_PATH,
            **custom_cache_result["summary"],
        }
    )


#### Load The Active Cache Metadata

After either path above, this shared cell loads the cache metadata and a small processed-file summary. The downstream sections use this same context regardless of whether the cache came from the workshop data or from your own CSV files.


In [ ]:
cache_context = load_session1_cache_context(
    cache_roots=[Path(CACHE_DIR), Path(DATASET_CONFIG['READ_CACHE_DIR'])],
    cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
)

# CACHE_STATE exposes the active cache paths, metadata, and row-level parquet parts.
CACHE_STATE = cache_context["notebook_values"]
active_cache_paths = CACHE_STATE["active_cache_paths"]
CACHE_DIR = CACHE_STATE["CACHE_DIR"]
ROW_CACHE_DIR = CACHE_STATE["ROW_CACHE_DIR"]
WINDOW_CACHE_PATH = CACHE_STATE["WINDOW_CACHE_PATH"]
METADATA_PATH = CACHE_STATE["METADATA_PATH"]
row_cache_dir = CACHE_STATE["row_cache_dir"]
window_cache_path = CACHE_STATE["window_cache_path"]
metadata_path = CACHE_STATE["metadata_path"]
metadata = CACHE_STATE["metadata"]
part_paths = CACHE_STATE["part_paths"]
part_to_source = CACHE_STATE["part_to_source"]
source_to_row_part = CACHE_STATE["source_to_row_part"]
print(cache_context["cache_summary"])
if not cache_context["processed_file_summary"].empty:
    display(cache_context["processed_file_summary"].head(8))


### Reusable Prep Utilities

The prep code is split by job so the same pieces are useful outside this workshop:

- `scripts/parquet_cache_utils.py` contains the generic CSV-to-parquet path.
- `csv_files_to_row_parquet_cache(...)` converts one or more ordinary CSV files into row-level parquet parts.
- `resolve_or_create_parquet_cache(...)` reuses an existing cache or builds one when requested.
- `scripts/onc_scalar_cache_utils.py` contains ONC-specific scalar prep.
- `create_onc_row_level_parquet_cache(...)` reads ONC CSV exports, types rows, and aligns companion device streams when needed.
- `create_onc_window_summary_parquet_cache(...)` builds fixed-window summaries from row-level parquet for clustering.

If you are bringing your own CSVs, start with `csv_files_to_row_parquet_cache(...)`. Use `header="first_row"` for ordinary CSVs, `header="auto"` for ONC-style metadata headers, or an integer such as `header=5` when the header starts on a known zero-based row.


---


### ⚡ Why The Parquet Cache Helps

Raw CSV is the source format. Parquet is the reusable analysis format.

This section shows four things:

- which cache bundle is active,
- which columns this notebook will read from the row-level cache,
- how the row-level and window-summary caches differ,
- how read time changes when we read all columns versus only the selected notebook columns.

For small, compact CSV samples, a full parquet read may not beat CSV. The practical benefit is that parquet keeps typed columns and supports **column-pruned reads**: loading only the timestamp, source file, target label, optional QC/context fields, and measurement columns needed by the next analysis step.


In [ ]:
cache_context = show_session1_cache_inspection(
    raw_data_dir=RAW_DATA_DIR,
    runtime_cache_dir=INTRO_SETUP["RUNTIME_CACHE_DIR"],
    read_cache_dir=DATASET_CONFIG['READ_CACHE_DIR'],
    cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
    target_flag=DATASET_CONFIG['TARGET_FLAG'],
    measurement_columns=DATASET_CONFIG['MEASUREMENT_COLUMNS'],
    optional_qc_columns=DATASET_CONFIG['OPTIONAL_QC_COLUMNS'],
    plot_measurement_column=DATASET_CONFIG['PLOT_MEASUREMENT_COLUMN'],
    plot_secondary_column=DATASET_CONFIG['PLOT_SECONDARY_COLUMN'],
)

# CACHE_INSPECTION_STATE records the resolved cache files and active column lists.
CACHE_INSPECTION_STATE = cache_context["notebook_values"]

# Keep dataset column settings current after inspection without unpacking each
# value into a separate notebook variable.
DATASET_CONFIG = {
    **DATASET_CONFIG,
    **{
        key: CACHE_INSPECTION_STATE[key]
        for key in [
            "MEASUREMENT_COLUMNS",
            "OPTIONAL_QC_COLUMNS",
            "PLOT_MEASUREMENT_COLUMN",
            "PLOT_SECONDARY_COLUMN",
            "WINDOW_FEATURE_COLUMNS",
            "ROW_USE_COLUMNS",
            "WINDOW_USE_COLUMNS",
        ]
    },
}


In [ ]:
PARQUET_READ_SAMPLE_ROWS = 50_000

cache_read_result = show_session1_cache_read_comparison(
    representative_raw_path=CACHE_INSPECTION_STATE["representative_raw_path"],
    representative_parquet_path=CACHE_INSPECTION_STATE["representative_parquet_path"],
    row_use_columns=DATASET_CONFIG['ROW_USE_COLUMNS'],
    sample_rows=PARQUET_READ_SAMPLE_ROWS,
)


### Choosing A Useful Storage Format

Parquet is a strong fit for **typed tabular data** and many **1D time-series** workflows, but it is not the only good option.

A simple rule of thumb is: store the data in a format that matches the natural unit of your experiment.

- **Scalar tables / 1D time series**: parquet is often ideal because you can read only the columns you need.
- **Sequences that will be windowed later**: row-level parquet still works well because you can build windows on demand.
- **Images or 2D products** such as spectrograms: a folder of image files can be perfectly reasonable if each file is already one example.
- **Large 2D or 3D numeric arrays**: formats like NumPy `.npy` / `.npz`, HDF5, Zarr, or NetCDF are often a better fit than CSV.
- **Text examples**: JSONL and parquet are common because each row can hold one document plus labels and metadata.

So the real lesson is not "always use parquet." It is "pick a reusable format that preserves the structure of your examples and is efficient for the experiments you plan to run."


## Part 3 — Understanding and Defining the Reviewed Modelling Dataset

This part turns the prepared parquet cache into the data the models will use.

In this notebook, the **reviewed modelling dataset** means: the rows with usable reviewed labels, after any row-budget cap from `DATA_FRACTION`, plus the model target columns we create from those labels.

The **fixed split** means: the train / validation / test division that stays unchanged while we compare different models and train-only sampling choices.

The code uses the same vocabulary: `reviewed_model_df` is the labelled row dataset used for modelling, and `FIXED_SPLIT_STRATEGY` describes how those rows are divided into train, validation, and test.


### What This Part Is For

Now that the data is cleaned and stored efficiently, the next job is to define the reviewed modelling dataset and fixed evaluation split.

In this part you will:

- choose which prepared parquet files to inspect,
- keep only the reviewed rows,
- compare train / validation / test split strategies,
- lock one fixed split,
- then optionally shrink only the training split.

A lot of the clarity in an ML workflow comes from this dataset-definition step, not just from the model choice later on.


### Phase 1 — Define The Reviewed Modelling Dataset

In this first phase, we use the prepared parquet cache to understand the reviewed data, compare fixed split strategies, and lock the train / validation / test split before we make any train-only runtime decisions.


---


In [ ]:
cache_context = load_session1_cache_context(
    cache_roots=[Path(CACHE_DIR), Path(DATASET_CONFIG['READ_CACHE_DIR'])],
    cache_bundle_name=DATASET_CONFIG['CACHE_BUNDLE_NAME'],
)

# CACHE_STATE exposes the active cache paths, metadata, and row-level parquet parts.
CACHE_STATE = cache_context["notebook_values"]
active_cache_paths = CACHE_STATE["active_cache_paths"]
CACHE_DIR = CACHE_STATE["CACHE_DIR"]
ROW_CACHE_DIR = CACHE_STATE["ROW_CACHE_DIR"]
WINDOW_CACHE_PATH = CACHE_STATE["WINDOW_CACHE_PATH"]
METADATA_PATH = CACHE_STATE["METADATA_PATH"]
row_cache_dir = CACHE_STATE["row_cache_dir"]
window_cache_path = CACHE_STATE["window_cache_path"]
metadata_path = CACHE_STATE["metadata_path"]
metadata = CACHE_STATE["metadata"]
part_paths = CACHE_STATE["part_paths"]
part_to_source = CACHE_STATE["part_to_source"]
source_to_row_part = CACHE_STATE["source_to_row_part"]
print(
    {
        **cache_context["cache_summary"],
        "runtime_output_root": str(INTRO_SETUP["RUNTIME_OUTPUT_ROOT"]),
        "model_output_dir": str(INTRO_SETUP["MODEL_OUTPUT_DIR"]),
        "task_mode": DATASET_CONFIG['TASK_MODE'],
    }
)


In [ ]:
# Show the source files represented in the active parquet cache.
processed_file_summary = cache_context["processed_file_summary"]
if processed_file_summary.empty:
    print("This cache metadata does not include per-file summaries.")
else:
    display(processed_file_summary.head(8))


#### 🚚 Use The Prepared Cache To Understand The Dataset

Parquet gives us fast access to the cleaned dataset, so we can inspect prepared data directly with only the columns we need.

In this part we will:

- load the prepared parquet cache,
- summarise **reviewed rows**, which have labels in `GOOD_LABELS` or `ISSUE_LABELS`,
- keep track of **unreviewed rows**, which are present in the cache but do not have labels we use for supervised modelling,
- check how the flags change across time,
- compare split strategies on the reviewed rows,
- then choose one fixed split.


#### 🎚️ Make The Row Budgets Visible

Large reviewed datasets can make live notebook runs slow. This section shows the row caps that affect the reviewed modelling dataset and the train/validation/test split setup.

`DATA_FRACTION` is the single top-level size control. Set `USE_DATA_FRACTION_BUDGETS = False` in the next cell to keep all reviewed rows for the dataset-building steps. Model-specific runtime caps, such as Random Forest fit rows or k-means window rows, are defined later at the start of the section that uses them.

| Cap variable | Where it is used | How it is computed when caps are on | Why it exists |
|---|---|---|---|
| `REVIEWED_MODEL_ROW_LIMIT` | reviewed modelling dataframe | `FULL_REVIEWED_ROW_COUNT * DATA_FRACTION` | build a smaller reviewed dataframe before split comparison |
| `TRAIN_SUBSET_MAX_ROWS` | train-only subset comparison | `TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION`, with a minimum | compare train-subsampling strategies quickly |
| `ISSUE_ROWS_PER_FILE` | `issue_focused` train subset | `ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION`, with a minimum | control how many extra issue rows are added |


In [ ]:
# Use every parquet part from the prepared cache. DATA_FRACTION limits rows later; it does not choose files.
TEMPORAL_SUMMARY_BIN_COUNT = 24
selected_paths = part_paths

# Keep source-file names handy for summaries and cache inspection plots.
selected_source_files = {
    part_to_source.get(path.name, path.name.replace(".parquet", ".csv"))
    for path in selected_paths
}

# Turn this off to keep every reviewed row during dataset-building steps.
# Model-specific row/window caps are defined later, inside the section that uses each model.
USE_DATA_FRACTION_BUDGETS = True
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Count usable reviewed rows: labels in GOOD_LABELS are normal, and labels in ISSUE_LABELS are issues.
target_distribution = {int(label): int(count) for label, count in metadata.get("target_distribution", {}).items()}
if not target_distribution:
    # Custom caches may not store label counts in metadata, so read only the target column.
    # This avoids loading every feature column just to count labels.
    target_count_df = load_full_row_level_frame(selected_paths, columns=[DATASET_CONFIG["TARGET_FLAG"]])
    target_distribution = {
        int(label): int(count)
        for label, count in target_count_df[DATASET_CONFIG["TARGET_FLAG"]].dropna().astype(int).value_counts().items()
    }

FULL_REVIEWED_ROW_COUNT = sum(
    target_distribution.get(int(label), 0)
    for label in [*DATASET_CONFIG["GOOD_LABELS"], *DATASET_CONFIG["ISSUE_LABELS"]]
)

# Base caps are compute budgets for live runs, not statistical rules.
# DATA_FRACTION scales them down for quick runs; the minimums prevent tiny examples.
TRAIN_SUBSET_BASE_ROWS = 1_000_000  # Max train rows used when comparing train-only subsampling strategies.
TRAIN_SUBSET_MIN_ROWS = 10_000      # Smallest train-subset comparison size allowed after scaling.
ISSUE_ROWS_PER_FILE_BASE = 12_000   # Extra issue rows requested by the issue_focused train subset.
ISSUE_ROWS_PER_FILE_MIN = 1_000     # Smallest issue-focused add-on after scaling.

if USE_DATA_FRACTION_BUDGETS and DATA_FRACTION < 0.999:
    BUDGET_MODE = "DATA_FRACTION dataset/split caps"
    BUDGET_DATA_FRACTION = DATA_FRACTION
    REVIEWED_MODEL_ROW_LIMIT = max(1, int(round(FULL_REVIEWED_ROW_COUNT * DATA_FRACTION)))
    TRAIN_SUBSET_MAX_ROWS = max(TRAIN_SUBSET_MIN_ROWS, int(TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION))
    ISSUE_ROWS_PER_FILE = max(ISSUE_ROWS_PER_FILE_MIN, int(ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION))
else:
    BUDGET_MODE = "full reviewed data / no dataset row caps"
    BUDGET_DATA_FRACTION = 1.0
    REVIEWED_MODEL_ROW_LIMIT = FULL_REVIEWED_ROW_COUNT
    TRAIN_SUBSET_MAX_ROWS = None
    ISSUE_ROWS_PER_FILE = ISSUE_ROWS_PER_FILE_BASE

# Variables created here and used later:
# - REVIEWED_MODEL_ROW_LIMIT limits the reviewed modelling dataframe before split comparison.
# - TRAIN_SUBSET_MAX_ROWS limits train-only subsampling comparisons; None means use the full train split.
# - ISSUE_ROWS_PER_FILE controls how many issue rows issue_focused sampling can add from each source file.

#### 🧰 Split And Subsampling Helpers

The next cell defines the helper functions used to build the fixed split and the train-only subsets.

The split helpers answer: “which reviewed rows belong in train, validation, and test?”

- `global_contiguous` makes one early/middle/late split over the full time axis.
- `per_source_contiguous` makes that early/middle/late split separately inside each source file.
- `interleaved_blocks` rotates fixed-size time blocks across train, validation, and test.

The train-subset helpers answer a separate question: “after the split is fixed, should we train on all training rows or a smaller training-only subset?”

- `time_spread` keeps broad time coverage.
- `issue_focused` keeps broad coverage and adds more issue rows.
- `balanced_reviewed` aims for a chosen good-versus-issue mix when issues are rare.

These functions are kept visible because participants may want to copy and modify them for their own split or sampling strategy.


In [ ]:
# These helpers define the split and train-only sampling logic used in Part 3.
# Each function focuses on one idea so you can trace how the rows move through the workflow.

def evenly_spaced_take(frame: pd.DataFrame, limit: int | None, *, time_column: str = "Time UTC") -> pd.DataFrame:
    """Return up to `limit` rows spread across the dataframe's full time range.

    Use `time_column="Time UTC"` for row-level data and
    `time_column="window_start"` for cached window-summary data.
    """
    # Sort by the relevant time column so the sample keeps chronological order.
    sort_column = time_column if time_column in frame.columns else "window_start" if "window_start" in frame.columns else None
    ordered = frame.sort_values(sort_column).reset_index(drop=True).copy() if sort_column else frame.reset_index(drop=True).copy()
    # If we do not need to shrink the dataframe, return the full ordered version.
    if limit is None or len(ordered) <= limit:
        return ordered
    # Pick row positions spread across the whole time range instead of keeping only the earliest rows.
    indices = np.linspace(0, len(ordered) - 1, num=limit, dtype=int)
    return ordered.iloc[indices].reset_index(drop=True)


def split_global_contiguous(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
) -> dict[str, pd.DataFrame]:
    """Split one dataframe into early train, middle validation, and late test segments.

    Use this when you want each split to cover a different part of the full
    fixed-split timeline.
    """
    # This strategy makes one early / middle / late cut across the full time axis.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    train_cut = int(len(ordered) * train_fraction)
    validation_cut = int(len(ordered) * (train_fraction + validation_fraction))
    return {
        "train": ordered.iloc[:train_cut].reset_index(drop=True),
        "validation": ordered.iloc[train_cut:validation_cut].reset_index(drop=True),
        "test": ordered.iloc[validation_cut:].reset_index(drop=True),
    }


def split_per_source_contiguous(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    source_column: str = "source_file",
) -> dict[str, pd.DataFrame]:
    """Apply the contiguous time split within each source file, then recombine the results.

    This lets every source contribute rows to train, validation, and test while
    still preserving time order inside each file.
    """
    # If we do not know which file each row came from, fall back to one global time split.
    if source_column not in frame.columns:
        return split_global_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )

    # Split each source file separately so every file can contribute rows to every split.
    split_frames: dict[str, list[pd.DataFrame]] = {"train": [], "validation": [], "test": []}
    for _, source_frame in frame.groupby(source_column, sort=False, observed=False):
        local_split = split_global_contiguous(
            source_frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
        for split_name, split_frame in local_split.items():
            split_frames[split_name].append(split_frame)

    # Concatenate the per-file slices back together and restore one global time ordering.
    empty_template = frame.iloc[0:0].copy()
    return {
        split_name: (
            pd.concat(split_frames[split_name], ignore_index=True).sort_values("Time UTC").reset_index(drop=True)
            if split_frames[split_name]
            else empty_template.copy()
        )
        for split_name in ["train", "validation", "test"]
    }


def split_interleaved_blocks(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    block_rows: int,
    source_column: str = "source_file",
) -> dict[str, pd.DataFrame]:
    """Rotate fixed-size time blocks across train, validation, and test splits.

    This mixes time periods more than one large contiguous cut while still
    keeping nearby rows together inside each block.
    """
    if block_rows <= 0:
        raise ValueError("block_rows must be positive")

    # Start from time-ordered rows so each block represents one local stretch of time.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if source_column not in ordered.columns:
        ordered_groups = [(None, ordered)]
    else:
        ordered_groups = list(ordered.groupby(source_column, sort=False, observed=False))

    # The cycle decides how often each split appears as we rotate through blocks.
    cycle_length = 20
    train_slots = max(1, int(round(train_fraction * cycle_length)))
    validation_slots = max(1, int(round(validation_fraction * cycle_length)))
    if train_slots + validation_slots >= cycle_length:
        validation_slots = max(1, cycle_length - train_slots - 1)

    # Walk through the files in order, cut each one into fixed-size blocks, and rotate those blocks across splits.
    split_frames: dict[str, list[pd.DataFrame]] = {"train": [], "validation": [], "test": []}
    block_index = 0
    for _, source_frame in ordered_groups:
        source_frame = source_frame.sort_values("Time UTC").reset_index(drop=True)
        for start in range(0, len(source_frame), block_rows):
            block = source_frame.iloc[start : start + block_rows].copy()
            if block.empty:
                continue
            slot = block_index % cycle_length
            if slot < train_slots:
                split_name = "train"
            elif slot < train_slots + validation_slots:
                split_name = "validation"
            else:
                split_name = "test"
            split_frames[split_name].append(block)
            block_index += 1

    # Recombine the assigned blocks and restore chronological order inside each split.
    empty_template = ordered.iloc[0:0].copy()
    return {
        split_name: (
            pd.concat(split_frames[split_name], ignore_index=True).sort_values("Time UTC").reset_index(drop=True)
            if split_frames[split_name]
            else empty_template.copy()
        )
        for split_name in ["train", "validation", "test"]
    }


def split_reviewed_frame(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    strategy: str,
    block_rows: int = 1024,
) -> dict[str, pd.DataFrame]:
    """Run the selected fixed split strategy and return the three split dataframes.

    Later cells call this helper so the strategy choice lives in one place.
    """
    # This wrapper keeps the later notebook cells simple: they only need to name the strategy they want.
    if strategy == "global_contiguous":
        return split_global_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
    if strategy == "per_source_contiguous":
        return split_per_source_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
    if strategy == "interleaved_blocks":
        return split_interleaved_blocks(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
            block_rows=block_rows,
        )
    raise ValueError(f"Unsupported split strategy: {strategy}")


def subsample_time_spread(frame: pd.DataFrame, rows_limit: int | None) -> pd.DataFrame:
    """Shrink the training set by keeping rows spread across the full training time range.

    rows_limit is the maximum number of rows to return. Use None to keep the
    full input frame.
    """
    # Keep rows spread across the full time range while shrinking the training set.
    return evenly_spaced_take(frame, rows_limit)


def subsample_issue_focused(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    target_column: str,
    issue_labels: list[int] | tuple[int, ...],
    issue_rows: int,
) -> pd.DataFrame:
    """Keep broad time coverage, then add extra reviewed issue rows to the subset.

    rows_limit is the final maximum number of rows to return. issue_rows is how
    many issue-labelled rows we try to add before trimming back to rows_limit.
    Use this when issue examples are rare and you want a smaller training set to
    include more of them.
    """
    # Sort first so the subset keeps a readable time ordering.
    work = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if rows_limit is None or len(work) <= rows_limit:
        return work

    # Keep a broad time-spread sample, then add extra issue examples on top of it.
    normalized_issue_labels = [int(label) for label in issue_labels]
    dedupe_columns = [column for column in ["source_file", "Time UTC"] if column in work.columns]

    base_limit = max(int(rows_limit) - int(issue_rows), 0)
    sampled_frame = evenly_spaced_take(work, base_limit)
    issue_mask = pd.to_numeric(work[target_column], errors="coerce").isin(normalized_issue_labels)
    issue_frame = work.loc[issue_mask].reset_index(drop=True)
    issue_sample = evenly_spaced_take(issue_frame, int(issue_rows)) if int(issue_rows) > 0 else issue_frame.iloc[0:0].copy()

    # Drop duplicates because some rows may appear in both the base sample and the issue-focused sample.
    sampled_frame = pd.concat([sampled_frame, issue_sample], ignore_index=True)
    if dedupe_columns:
        sampled_frame = sampled_frame.drop_duplicates(subset=dedupe_columns)
    else:
        sampled_frame = sampled_frame.drop_duplicates()
    sampled_frame = sampled_frame.sort_values("Time UTC").reset_index(drop=True)
    return evenly_spaced_take(sampled_frame, rows_limit)


def subsample_balanced_reviewed(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    target_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    balanced_issue_share: float,
) -> pd.DataFrame:
    """Build a smaller training set with a chosen good-versus-issue class mix.

    rows_limit is the final maximum number of rows to return. The requested
    issue share is a target, not a guarantee. If the full training split
    contains fewer issue rows than the target asks for, this helper keeps all
    available issue rows and fills the rest of the row budget with good rows.
    """
    # Sort first so the returned training set still reads in time order.
    work = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if rows_limit is None or len(work) <= rows_limit:
        return work
    if not 0 < balanced_issue_share < 1:
        raise ValueError("balanced_issue_share must be in the open interval (0, 1)")

    # Split the reviewed rows into good vs issue groups, then sample each group to the requested share.
    normalized_good_labels = [int(label) for label in good_labels]
    normalized_issue_labels = [int(label) for label in issue_labels]
    numeric_target = pd.to_numeric(work[target_column], errors="coerce")
    good_frame = work.loc[numeric_target.isin(normalized_good_labels)].reset_index(drop=True)
    issue_frame = work.loc[numeric_target.isin(normalized_issue_labels)].reset_index(drop=True)
    dedupe_columns = [column for column in ["source_file", "Time UTC"] if column in work.columns]

    def drop_selected_duplicates(candidate_frame: pd.DataFrame) -> pd.DataFrame:
        if dedupe_columns:
            return candidate_frame.drop_duplicates(subset=dedupe_columns)
        return candidate_frame.drop_duplicates()

    def remove_already_selected(candidate_frame: pd.DataFrame, selected_frame: pd.DataFrame) -> pd.DataFrame:
        if selected_frame.empty:
            return candidate_frame
        if dedupe_columns:
            selected_keys = set(map(tuple, selected_frame[dedupe_columns].to_numpy()))
            keep_mask = ~candidate_frame[dedupe_columns].apply(tuple, axis=1).isin(selected_keys)
            return candidate_frame.loc[keep_mask]
        return candidate_frame.drop(selected_frame.index, errors="ignore")

    requested_issue_rows = min(max(int(round(rows_limit * balanced_issue_share)), 0), int(rows_limit))
    issue_target = min(requested_issue_rows, len(issue_frame))
    good_target = int(rows_limit) - issue_target

    # Keep as many issue rows as the target allows. If issues are scarce, keep all of them.
    issue_sample = evenly_spaced_take(issue_frame, issue_target)
    good_sample = evenly_spaced_take(good_frame, good_target)

    sampled_frame = pd.concat([good_sample, issue_sample], ignore_index=True)
    sampled_frame = drop_selected_duplicates(sampled_frame).sort_values("Time UTC").reset_index(drop=True)

    # If one group ran out of rows, refill from rows that have not already been selected.
    if len(sampled_frame) < rows_limit:
        fill_needed = int(rows_limit) - len(sampled_frame)
        fill_candidates = remove_already_selected(work, sampled_frame).reset_index(drop=True)
        fill_frame = evenly_spaced_take(fill_candidates, fill_needed)
        sampled_frame = pd.concat([sampled_frame, fill_frame], ignore_index=True)
        sampled_frame = drop_selected_duplicates(sampled_frame).sort_values("Time UTC").reset_index(drop=True)

    return sampled_frame.iloc[: int(rows_limit)].sort_values("Time UTC").reset_index(drop=True)


def subsample_train_frame(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    strategy: str,
    target_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    issue_rows: int,
    balanced_issue_share: float,
) -> pd.DataFrame:
    """Apply the requested train-only subsampling rule and return the result.

    rows_limit is the maximum number of training rows to keep. Validation and
    test stay unchanged; only the training split is reduced here.
    """
    # Sort once up front so every strategy starts from the same time-ordered training frame.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    # "full_train" means keep the complete training split without further shrinking.
    if rows_limit is None or len(ordered) <= rows_limit or strategy == "full_train":
        return ordered
    # Otherwise dispatch to the selected subsampling rule.
    if strategy == "time_spread":
        return subsample_time_spread(ordered, rows_limit)
    if strategy == "issue_focused":
        return subsample_issue_focused(
            ordered,
            rows_limit=rows_limit,
            target_column=target_column,
            issue_labels=issue_labels,
            issue_rows=issue_rows,
        )
    if strategy == "balanced_reviewed":
        return subsample_balanced_reviewed(
            ordered,
            rows_limit=rows_limit,
            target_column=target_column,
            good_labels=good_labels,
            issue_labels=issue_labels,
            balanced_issue_share=balanced_issue_share,
        )
    raise ValueError(f"Unsupported train subset strategy: {strategy}")


#### Load A Narrow Label Overview

The split and sampling helper functions are now available. The next code cell reads a small overview dataframe with just the timestamp, source file, and active target label. That overview is enough to count labels, inspect time coverage, and design train/validation/test splits before the modelling sections load the full measurement feature table.

In [ ]:
# Load only the columns needed for label accounting and split design.
overview_columns = ["Time UTC", "source_file", DATASET_CONFIG['TARGET_FLAG']]
dataset_label_df = load_full_row_level_frame(selected_paths, columns=overview_columns)
dataset_label_df[DATASET_CONFIG['TARGET_FLAG']] = pd.to_numeric(dataset_label_df[DATASET_CONFIG['TARGET_FLAG']], errors="coerce")
if "source_file" in dataset_label_df.columns:
    dataset_label_df["source_file"] = dataset_label_df["source_file"].astype("category")

print(
    {
        "selected_parts": len(selected_paths),
        "selected_rows": len(dataset_label_df),
        "overview_columns_loaded": overview_columns,
    }
)


#### Count Reviewed Rows Before Splitting

The next code cell separates two related ideas: every usable reviewed row, and the smaller reviewed modelling dataframe used for the live notebook. `GOOD_LABELS` and `ISSUE_LABELS` define which labels count as reviewed, while `REVIEWED_MODEL_ROW_LIMIT` applies the visible `DATA_FRACTION` row budget. The returned dataframes are reused below for time summaries, split comparisons, and the final selected train/validation/test split.

In [ ]:
reviewed_model_accounting = show_reviewed_model_row_accounting(
    dataset_label_df,
    target_flag=DATASET_CONFIG['TARGET_FLAG'],
    task_mode=DATASET_CONFIG['TASK_MODE'],
    good_labels=DATASET_CONFIG['GOOD_LABELS'],
    issue_labels=DATASET_CONFIG['ISSUE_LABELS'],
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,  # Row cap computed in the row-budget cell above; None would keep all reviewed rows.
    data_fraction=BUDGET_DATA_FRACTION,
    flag_palette=DEFAULT_FLAG_PALETTE,
    label_meanings=DATASET_CONFIG['FLAG_EXAMPLE_LABEL_MEANINGS'],
)

# reviewed_label_df keeps every usable reviewed row.
# reviewed_model_df applies DATA_FRACTION and is the dataframe used for split comparison below.
reviewed_label_df = reviewed_model_accounting["reviewed_label_df"]
reviewed_model_df = reviewed_model_accounting["reviewed_model_df"]
active_labels = reviewed_model_accounting["active_labels"]
model_good_labels = reviewed_model_accounting["model_good_labels"]
model_issue_labels = reviewed_model_accounting["model_issue_labels"]
selected_target_counts = reviewed_model_accounting["selected_target_counts"]
selected_reviewed_counts = reviewed_model_accounting["selected_reviewed_counts"]
reviewed_model_counts = reviewed_model_accounting["reviewed_model_counts"]
row_accounting_summary = reviewed_model_accounting["row_summary"]
REVIEWED_MODEL_ROW_LIMIT = reviewed_model_accounting["summary"]["reviewed_model_row_limit"]


#### Verify How The Flags Change Across Time

Before choosing a split strategy, look at **where the reviewed modelling labels live on the time axis**.

This is an important dataset check because:

- issue labels may cluster into only a few time periods,
- the label mix may shift across time,
- and those patterns directly affect how a split behaves.

These plots provide a dataset sanity check before we define train / validation / test.

One plot shows the reviewed modelling label mix through time. Another removes the good reviewed rows so you can focus only on how the issue labels are distributed.


In [ ]:
temporal_summary_bundle = show_temporal_flag_summary(
    reviewed_model_df,
    target_flag=DATASET_CONFIG['TARGET_FLAG'],
    selected_path_count=len(selected_paths),
    temporal_summary_bin_count=TEMPORAL_SUMMARY_BIN_COUNT,
    good_labels=DATASET_CONFIG['GOOD_LABELS'],
    issue_labels=DATASET_CONFIG['ISSUE_LABELS'],
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=DATASET_CONFIG['FLAG_EXAMPLE_DISPLAY_NAME'],
)
temporal_bin_count = temporal_summary_bundle["bin_count"]
temporal_labels = temporal_summary_bundle["labels"]
temporal_counts = temporal_summary_bundle["counts"]
temporal_shares = temporal_summary_bundle["shares"]
temporal_issue_only_shares = temporal_summary_bundle["issue_only_shares"]
temporal_summary = temporal_summary_bundle["summary"]

print(
    {
        "temporal_bin_count": temporal_bin_count,
        "time_bins_shown": len(temporal_summary),
        "issue_labels": DATASET_CONFIG['ISSUE_LABELS'],
        "max_issue_share_pct": float(temporal_summary["issue_share_pct"].max()) if len(temporal_summary) else 0.0,
        "min_issue_share_pct": float(temporal_summary["issue_share_pct"].min()) if len(temporal_summary) else 0.0,
    }
)


#### Split Strategy Controls

These settings control the split comparison in the next cells.

- `global_contiguous`: one early / middle / late cut across the whole time axis.
- `per_source_contiguous`: make an early / middle / late cut inside each source file, then combine the pieces.
- `interleaved_blocks`: keep short local time blocks intact, but distribute those blocks across train / validation / test.

`INTERLEAVED_BLOCK_ROWS` controls how large each local block is in the interleaved strategy.

Validation and test stay unbalanced so they reflect the natural distribution produced by the split.


In [ ]:
SUPPORTED_SPLIT_STRATEGIES = (
    "global_contiguous",
    "per_source_contiguous",
    "interleaved_blocks",
)
INTERLEAVED_BLOCK_ROWS = 1024

show_setup_json(
    {
        "SUPPORTED_SPLIT_STRATEGIES": SUPPORTED_SPLIT_STRATEGIES,
        "INTERLEAVED_BLOCK_ROWS": INTERLEAVED_BLOCK_ROWS,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
    }
)


#### Compare Split Strategies

This comparison uses the **full reviewed rows** from the selected parquet parts.

That is deliberate: we want the split decision to be based on the real reviewed dataset, not on a tiny notebook sample.

The table below keeps the comparison simple:

- rows per split,
- issue rows and issue share per split,
- and the time span covered by each split.


In [ ]:
comparison_labels = [
    label
    for label in sorted(set(model_good_labels).union(model_issue_labels))
    if label in set(reviewed_model_df["model_target"].dropna().astype(int).unique().tolist())
]
present_issue_labels = [label for label in model_issue_labels if label in comparison_labels]

split_strategy_labels = {
    "global_contiguous": "Global contiguous",
    "per_source_contiguous": "Per-source contiguous",
    "interleaved_blocks": "Interleaved blocks",
}

strategy_frames_by_name = {}
strategy_comparison_rows = []

for strategy_name in SUPPORTED_SPLIT_STRATEGIES:
    strategy_frames = split_reviewed_frame(
        reviewed_model_df,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        strategy=strategy_name,
        block_rows=INTERLEAVED_BLOCK_ROWS,
    )
    strategy_frames_by_name[strategy_name] = strategy_frames

    for split_name, split_frame in strategy_frames.items():
        split_labels = pd.to_numeric(split_frame["model_target"], errors="coerce").dropna().astype(int)
        label_counts = split_labels.value_counts().sort_index()
        issue_rows = int(label_counts.reindex(present_issue_labels, fill_value=0).sum())
        strategy_comparison_rows.append(
            {
                "strategy": split_strategy_labels[strategy_name],
                "split": split_name,
                "rows": len(split_frame),
                "issue_rows": issue_rows,
                "issue_share_pct": round(100 * issue_rows / max(len(split_frame), 1), 2),
                "time_start": split_frame["Time UTC"].min().isoformat() if len(split_frame) else None,
                "time_end": split_frame["Time UTC"].max().isoformat() if len(split_frame) else None,
            }
        )

strategy_comparison_df = pd.DataFrame(strategy_comparison_rows)
display(
    strategy_comparison_df.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
        }
    )
)


**Visual comparison**

Now that you have seen the labels through time, we can ask a more focused question: **how does the split itself change the balance?**

We compare the same three strategies on the same reviewed modelling dataset:

- **Global contiguous**: earliest rows in train, middle rows in validation, latest rows in test.
- **Per-source contiguous**: each source file contributes an early / middle / late slice.
- **Interleaved blocks**: short local time blocks stay intact, but the blocks rotate across splits.

The timeline plot shows where each split lands across the full selected time period. The composition plot then shows the reviewed modelling label mix for each strategy and, separately, the issue-label mix without the good rows.


In [ ]:
strategy_timeline_bundle = show_split_strategy_timeline(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    figure_title="Where each split strategy places train / validation / test over time",
)
strategy_timeline_summary = strategy_timeline_bundle["summary"]

strategy_plot_bundle = show_split_strategy_comparison(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    issue_labels=present_issue_labels,
    label_column="model_target",
    flag_palette=DEFAULT_FLAG_PALETTE,
    legend_title="model target",
    figure_title="How the split strategy changes label balance on the reviewed modelling dataset",
)
strategy_shares_by_name = strategy_plot_bundle["shares"]
strategy_issue_only_shares_by_name = strategy_plot_bundle["issue_only_shares"]

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "reviewed_model_rows": len(reviewed_model_df),
        "INTERLEAVED_BLOCK_ROWS": INTERLEAVED_BLOCK_ROWS,
        "present_issue_labels": present_issue_labels,
    }
)


**Questions to ask before you lock the split**

Before choosing a fixed split, pause and look for structure that a simple label-balance table cannot show on its own:

1. Are the issue flags isolated points, or do they cluster into longer episodes?
2. Could one real bad-data event be getting cut across train, validation, and test?
3. If the model uses windows or lagged context, would nearby timestamps leak information across split boundaries?
4. Does this split answer the question you care about: generalize to nearby conditions, or generalize to a completely new bad episode?

Keep those questions in mind as you pick the split strategy below. The advanced notebook comes back to them with a stronger episode-aware fixed split.


#### Choose And Build The Fixed Split

This is the single fixed train / validation / test split used by the model sections that follow.

The next cell keeps the editable choices visible. The utility helper handles the repetitive work: loading reviewed labels, creating the modelling target, applying the split, and returning the standard dataframe names used later.


In [ ]:
# FIXED_SPLIT_STRATEGY controls how reviewed rows are assigned to train / validation / test.
# "global_contiguous": split the whole time-ordered dataset into one early/middle/late split.
# "per_source_contiguous": split each source file separately, then combine the matching split pieces.
# "interleaved_blocks": alternate fixed-size time blocks across train / validation / test.
FIXED_SPLIT_STRATEGY = "global_contiguous"
if FIXED_SPLIT_STRATEGY not in SUPPORTED_SPLIT_STRATEGIES:
    raise ValueError(
        f"Unsupported split strategy: {FIXED_SPLIT_STRATEGY}. Choose from {SUPPORTED_SPLIT_STRATEGIES}."
    )

# REVIEWED_MODEL_ROW_LIMIT was computed from DATA_FRACTION and the usable reviewed row count above.
# With DATA_FRACTION = 1.0, this equals the full usable reviewed count.

# FIXED_SPLIT_BLOCK_ROWS only changes interleaved splits.
# The contiguous strategies ignore block rows because they split by continuous time ranges.
FIXED_SPLIT_BLOCK_ROWS = INTERLEAVED_BLOCK_ROWS if FIXED_SPLIT_STRATEGY == "interleaved_blocks" else None

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "FIXED_SPLIT_STRATEGY": FIXED_SPLIT_STRATEGY,
        "FIXED_SPLIT_BLOCK_ROWS": FIXED_SPLIT_BLOCK_ROWS,
    }
)


In [ ]:
# Build the reviewed modelling table and fixed split in one utility call.
reviewed_split_bundle = build_reviewed_modelling_split(
    # selected_paths: row-level parquet parts selected from the active cache earlier in Part 3.
    selected_paths=selected_paths,
    # target_flag: the reviewed QC/custom label column we want to learn from.
    target_flag=DATASET_CONFIG['TARGET_FLAG'],
    # task_mode: "multiclass" keeps label values; "binary" converts to issue vs not-issue.
    task_mode=DATASET_CONFIG['TASK_MODE'],
    # good_labels / issue_labels: define which reviewed labels become usable modelling rows.
    good_labels=DATASET_CONFIG['GOOD_LABELS'],
    issue_labels=DATASET_CONFIG['ISSUE_LABELS'],
    # model_row_limit: optional row cap derived from DATA_FRACTION for faster live runs.
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    # train_fraction / validation_fraction: test receives the remaining fraction.
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    # split_strategy / split_block_rows: the fixed split choices from the control cell above.
    split_strategy=FIXED_SPLIT_STRATEGY,
    split_block_rows=FIXED_SPLIT_BLOCK_ROWS,
    # measurement_columns: kept with the returned metadata for later model sections.
    measurement_columns=DATASET_CONFIG['MEASUREMENT_COLUMNS'],
)

# SPLIT_STATE contains the dataframe names used by the ML sections.
SPLIT_STATE = reviewed_split_bundle["notebook_values"]
measurement_columns = SPLIT_STATE["measurement_columns"]
task_mode = SPLIT_STATE["task_mode"]
target_name = SPLIT_STATE["target_name"]
reviewed_model_label_columns = SPLIT_STATE["reviewed_model_label_columns"]
reviewed_model_df = SPLIT_STATE["reviewed_model_df"]
model_df = SPLIT_STATE["model_df"]
active_labels = SPLIT_STATE["active_labels"]
model_good_labels = SPLIT_STATE["model_good_labels"]
model_issue_labels = SPLIT_STATE["model_issue_labels"]
fixed_split_frames = SPLIT_STATE["fixed_split_frames"]
train_full_df = SPLIT_STATE["train_full_df"]
valid_df = SPLIT_STATE["valid_df"]
test_df = SPLIT_STATE["test_df"]
show_reviewed_modelling_split_build(reviewed_split_bundle)


#### Review The Fixed Split

The review helper shows the split sizes, issue coverage, time span, full label mix, and issue-only label mix. Validation and test are natural outputs of the split and preserve the label mix produced by that split strategy.


In [ ]:
fixed_split_review = show_fixed_split_review(
    # split_frames: the exact train / validation / test dataframes all model sections reuse.
    {"train": train_full_df, "validation": valid_df, "test": test_df},
    # issue_labels: which model_target values count as issue classes in the issue-only plot.
    issue_labels=model_issue_labels,
    # split_strategy_label: human-readable title for the plots and tables.
    split_strategy_label=split_strategy_labels.get(FIXED_SPLIT_STRATEGY, FIXED_SPLIT_STRATEGY),
    # flag_palette: consistent colours for label values across the notebook.
    flag_palette=DEFAULT_FLAG_PALETTE,
)

# Keep the displayed review objects available in case later cells or participants want to inspect them.
split_overview = fixed_split_review["overview"]
split_counts = fixed_split_review["counts"]
split_shares = fixed_split_review["shares"]
split_issue_only_shares = fixed_split_review["issue_only_shares"]


### Phase 2 — Design The Train-Only Subset

The reviewed modelling dataset is fixed now. From this point on, validation and test stay unchanged. The next decisions only affect **which training rows** we hand to the models.


#### Training Subset Comparison Settings

The fixed split is fixed now. `DATA_FRACTION` controls the train-only row budget used in the next block, while `TRAIN_SUBSET_STRATEGY` later chooses *which* training rows are kept.

`BALANCED_REVIEWED_TARGET_ISSUE_SHARE` only matters for `balanced_reviewed`. It is a target, not a guarantee: if the full training split does not contain enough issue rows, the helper keeps all available issue rows and fills the remaining row budget with good rows.

Validation and test stay untouched so the comparison stays fair.


In [ ]:
SUPPORTED_TRAIN_SUBSET_STRATEGIES = ("full_train", "time_spread", "balanced_reviewed", "issue_focused")

# TRAIN_SUBSET_MAX_ROWS is the maximum number of training rows the next comparison keeps.
# It was computed in the setup or row-budget cell from DATA_FRACTION and TRAIN_SUBSET_BASE_ROWS.
# Set TRAIN_SUBSET_MAX_ROWS = None here if you want the full training split.
# ISSUE_ROWS_PER_FILE only affects the "issue_focused" strategy: it is the number
# of extra issue-labelled rows that strategy asks for before trimming back to the row budget.
# Set ISSUE_ROWS_PER_FILE lower if the issue-focused subset is too dominated by rare labels.
#
# Example overrides:
# TRAIN_SUBSET_MAX_ROWS = None
# ISSUE_ROWS_PER_FILE = 2_000

BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25  # Used only by "balanced_reviewed"; 0.25 asks for 25% issue rows.
if not 0 < BALANCED_REVIEWED_TARGET_ISSUE_SHARE < 1:
    raise ValueError("BALANCED_REVIEWED_TARGET_ISSUE_SHARE must be in the interval (0, 1).")

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "ISSUE_ROWS_PER_FILE": ISSUE_ROWS_PER_FILE,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    }
)


#### Compare Train-Only Subset Strategies

These next cells compare alternative ways to choose rows from the training set while keeping the fixed validation and test splits as the held-out evaluation data.

First we summarise the candidate train-only subsets. Then we visualise how the target mix changes inside train.

In [ ]:
train_subset_preview_source = train_full_df.copy()
train_subset_rows_limit = (
    len(train_full_df)
    if TRAIN_SUBSET_MAX_ROWS is None
    else min(int(TRAIN_SUBSET_MAX_ROWS), len(train_full_df))
)

available_train_issue_rows = int(train_subset_preview_source["model_target"].isin(model_issue_labels).sum())
max_possible_balanced_issue_share = min(available_train_issue_rows, train_subset_rows_limit) / max(train_subset_rows_limit, 1)

train_subset_preview_frames = {"full_train": train_subset_preview_source}
for subset_strategy in ("time_spread", "balanced_reviewed", "issue_focused"):
    train_subset_preview_frames[subset_strategy] = subsample_train_frame(
        train_subset_preview_source,
        rows_limit=train_subset_rows_limit,
        strategy=subset_strategy,
        target_column="model_target",
        good_labels=model_good_labels,
        issue_labels=model_issue_labels,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )

train_subset_labels = {
    "full_train": "full train",
    "time_spread": "time-spread subset",
    "balanced_reviewed": (
        f"balanced-reviewed subset ({BALANCED_REVIEWED_TARGET_ISSUE_SHARE:.0%} target; "
        f"max {max_possible_balanced_issue_share:.1%})"
    ),
    "issue_focused": "issue-focused subset",
}

train_subset_overview_rows = []
for subset_name, subset_frame in train_subset_preview_frames.items():
    subset_counts = subset_frame["model_target"].value_counts().sort_index()
    issue_rows = int(subset_counts.reindex(model_issue_labels, fill_value=0).sum())
    train_subset_overview_rows.append(
        {
            "subset": train_subset_labels[subset_name],
            "rows": len(subset_frame),
            "issue_rows": issue_rows,
            "issue_share_pct": round(100 * issue_rows / max(len(subset_frame), 1), 2),
            "target_issue_share_pct": (
                round(100 * BALANCED_REVIEWED_TARGET_ISSUE_SHARE, 2)
                if subset_name == "balanced_reviewed"
                else None
            ),
            "max_possible_issue_share_pct": round(
                100 * min(available_train_issue_rows, len(subset_frame)) / max(len(subset_frame), 1),
                2,
            ),
            "time_start": subset_frame["Time UTC"].min().isoformat(),
            "time_end": subset_frame["Time UTC"].max().isoformat(),
        }
    )
train_subset_overview = pd.DataFrame(train_subset_overview_rows).set_index("subset")

train_budget_counts = pd.DataFrame(
    {
        subset_name: subset_frame["model_target"].value_counts().sort_index()
        for subset_name, subset_frame in train_subset_preview_frames.items()
    }
).fillna(0).astype(int)
train_budget_shares = train_budget_counts.div(train_budget_counts.sum(axis=0), axis=1).fillna(0.0)
display(
    train_subset_overview.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
            "target_issue_share_pct": "{:.2f}",
            "max_possible_issue_share_pct": "{:.2f}",
        },
        na_rep="-",
    )
)


In [ ]:
train_budget_plot = train_budget_shares.T.copy()
train_budget_plot.index = [
    f"{train_subset_labels[subset_name]} (n={int(train_budget_counts[subset_name].sum()):,})"
    for subset_name in train_budget_plot.index
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in train_budget_plot.columns]

fig, ax = plt.subplots(figsize=(11.8, 4.6))
train_budget_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.6)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in train")
ax.set_ylabel("")
ax.set_title("How the train-only subset strategy changes the training set")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="target_label", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()


#### Choose The Train-Only Subset

Pick the train-only subset strategy you want the models to fit on. This is the main knob to change when you want to see how train-set construction affects the models.

Validation and test stay fixed. This choice only affects the training rows that are handed to the models.


In [ ]:
# Change this directly to compare how different train-only subset choices affect the models.
# Options: "full_train", "time_spread", "balanced_reviewed", "issue_focused"
TRAIN_SUBSET_STRATEGY = "time_spread"
if TRAIN_SUBSET_STRATEGY not in SUPPORTED_TRAIN_SUBSET_STRATEGIES:
    raise ValueError(
        f"Unsupported TRAIN_SUBSET_STRATEGY: {TRAIN_SUBSET_STRATEGY}. Choose from {SUPPORTED_TRAIN_SUBSET_STRATEGIES}."
    )

train_df = subsample_train_frame(
    train_full_df,
    rows_limit=TRAIN_SUBSET_MAX_ROWS,
    strategy=TRAIN_SUBSET_STRATEGY,
    target_column="model_target",
    good_labels=model_good_labels,
    issue_labels=model_issue_labels,
    issue_rows=ISSUE_ROWS_PER_FILE,
    balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
)

show_setup_json(
    {
        "TRAIN_SUBSET_STRATEGY": TRAIN_SUBSET_STRATEGY,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
    }
)


## Next Step

Open `session1_machine_learning.ipynb` when you are ready to train models.

The ML notebook starts with one setup cell that rebuilds the same prepared dataset state automatically. If you want to change raw CSV paths, cache names, label columns, feature columns, row budgets, or split strategy, make those changes in that first ML setup cell.